In [2]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms


In [3]:
transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [5]:
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms)    
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=2)


100%|██████████| 170498071/170498071 [11:17<00:00, 251598.41it/s]


Extracting ./data\cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [11]:
image , lable =train_dataset[1]
image

tensor([[[ 0.2078, -0.0118, -0.1765,  ..., -0.2863, -0.3176, -0.3804],
         [ 0.0980,  0.1373, -0.0196,  ..., -0.2471, -0.3961, -0.4431],
         [ 0.0980,  0.0902, -0.0980,  ..., -0.3804, -0.4667, -0.4745],
         ...,
         [ 0.3725,  0.2235,  0.2078,  ..., -0.6706, -0.5216, -0.2706],
         [ 0.2941,  0.2235,  0.2471,  ..., -0.1922, -0.0353,  0.0275],
         [ 0.2784,  0.2392,  0.2784,  ...,  0.1216,  0.1216,  0.1216]],

        [[ 0.3882,  0.0745, -0.1843,  ..., -0.2549, -0.2941, -0.3647],
         [ 0.2549,  0.2000, -0.0196,  ..., -0.2235, -0.3725, -0.4275],
         [ 0.2157,  0.1451, -0.0980,  ..., -0.3569, -0.4510, -0.4588],
         ...,
         [ 0.3098,  0.2078,  0.2549,  ..., -0.7333, -0.5843, -0.3490],
         [ 0.2078,  0.1922,  0.2627,  ..., -0.2706, -0.1059, -0.0510],
         [ 0.1608,  0.1608,  0.2235,  ...,  0.0431,  0.0510,  0.0431]],

        [[ 0.4667,  0.0667, -0.2549,  ..., -0.4431, -0.4431, -0.4510],
         [ 0.3255,  0.2078, -0.0745,  ..., -0

In [12]:
image.size()

torch.Size([3, 32, 32])

In [8]:
class_name =['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']


In [14]:
class Neural_Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12 , 5)  #12, 28 ,28
        self.pool = nn.MaxPool2d(2, 2) #12, 14, 14
        self.conv2 =nn.Conv2d(12, 24, 5) #24, 10, 10  -> 24 ,5, 5 
        self.fc1 = nn.Linear(24 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x  = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [15]:
net =Neural_Net()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [18]:
for epoch in range(30):
    print(f'Tranning Epoch {epoch} ...')

    running_loss = 0.0

    for i,data in enumerate(train_loader):
        input,lable =data

        optimizer.zero_grad()

        outputs=net(input)

        loss=loss_function(outputs,lable)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')    

Tranning Epoch 0 ...
Loss: 1.7362
Tranning Epoch 1 ...
Loss: 1.5079
Tranning Epoch 2 ...
Loss: 1.3893
Tranning Epoch 3 ...
Loss: 1.2962
Tranning Epoch 4 ...
Loss: 1.2181
Tranning Epoch 5 ...
Loss: 1.1430
Tranning Epoch 6 ...
Loss: 1.0777
Tranning Epoch 7 ...
Loss: 1.0243
Tranning Epoch 8 ...
Loss: 0.9750
Tranning Epoch 9 ...
Loss: 0.9339
Tranning Epoch 10 ...
Loss: 0.8971
Tranning Epoch 11 ...
Loss: 0.8655
Tranning Epoch 12 ...
Loss: 0.8312
Tranning Epoch 13 ...
Loss: 0.7956
Tranning Epoch 14 ...
Loss: 0.7700
Tranning Epoch 15 ...
Loss: 0.7411
Tranning Epoch 16 ...
Loss: 0.7122
Tranning Epoch 17 ...
Loss: 0.6892
Tranning Epoch 18 ...
Loss: 0.6616
Tranning Epoch 19 ...
Loss: 0.6384
Tranning Epoch 20 ...
Loss: 0.6184
Tranning Epoch 21 ...
Loss: 0.5952
Tranning Epoch 22 ...
Loss: 0.5720
Tranning Epoch 23 ...
Loss: 0.5541
Tranning Epoch 24 ...
Loss: 0.5336
Tranning Epoch 25 ...
Loss: 0.5115
Tranning Epoch 26 ...
Loss: 0.4906
Tranning Epoch 27 ...
Loss: 0.4722
Tranning Epoch 28 ...
Loss: 0.

In [19]:
torch.save(net.state_dict(), 'first_classification_model.pth')

In [20]:
net = Neural_Net()
net.load_state_dict(torch.load('first_classification_model.pth'))

C:\Users\lapto\AppData\Local\Temp\ipykernel_13232\42112975.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load('first_classification_model.pth

<All keys matched successfully>

In [23]:
correct = 0 
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images,labels =data
        outputs=net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct+= (predicted == labels).sum().item()



accuracy =100 * correct /total

print(f'Accuracy : {accuracy}%')


Accuracy : 68.73%


In [27]:

from torchvision import transforms  
new_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

def load_image(image_path):
    image=Image.open(image_path)
    image= new_transform(image)
    image = image.unsqueeze(0)
    
    return image


image_path= [ '11.png','22.png','33.png', 'image.png']
images =[load_image(img) for img in image_path]

net.eval()

with torch.no_grad():
    for image in images:
        output=net(image)
        _,predected =torch.max(output ,1)
        print(f'Predected :{class_name[predected.item()]}')

Predected :cat
Predected :ship
Predected :ship
Predected :plane
